# Task 12 — Corrected stability-feature ablation

Evaluate the prespecified 61-feature MISFIT model and its ablations:

- 50 fold-fitted ESM2 local-delta principal components
- 7 structural features
- 4 stability-related features: `ddg_esm2`, `ddg_struct`, `ddg_rasp`, and `ddg_foldx`

The binary target is read directly from `Mislocalized`. All tables are joined using the canonical `KEY = Gene||Mutation_used`. Imputation, scaling, and PCA are fitted using training-fold data only.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE = Path("/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial")
FEATURES_CSV = BASE / "features_v3.csv"
ALPHAMISSENSE_CSV = BASE / "alphamissense_completed.csv"
OOF_CSV = BASE / "task12_ddg_ablation_oof.csv"
METRICS_CSV = BASE / "task12_ddg_ablation_metrics.csv"
IMPORTANCE_CSV = BASE / "task12_final_feature_importance.csv"

N_SPLITS = 5
RANDOM_STATE = 42
N_COMPONENTS = 50

ESM_COLS = [f"esm_{i}" for i in range(1280)]
STRUCT_COLS = [
    "plddt", "sasa", "rsa",
    "ss_helix", "ss_strand", "ss_coil",
    "delta_hydrophobicity",
]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]

CONFIG_FEATURES = {
    "baseline_pca_struct": [],
    "plus_ddg_esm2": ["ddg_esm2"],
    "plus_structure_ddgs": ["ddg_struct", "ddg_rasp", "ddg_foldx"],
    "final_all_ddgs": DDG_COLS,
}
CONFIG_DIMS = {name: N_COMPONENTS + len(STRUCT_COLS) + len(cols)
               for name, cols in CONFIG_FEATURES.items()}
FINAL_CONFIG = "final_all_ddgs"

print("Configurations:")
for name, columns in CONFIG_FEATURES.items():
    print(f"  {name:24s}: {CONFIG_DIMS[name]} features; stability={columns}")

Configurations:
  baseline_pca_struct     : 57 features; stability=[]
  plus_ddg_esm2           : 58 features; stability=['ddg_esm2']
  plus_structure_ddgs     : 60 features; stability=['ddg_struct', 'ddg_rasp', 'ddg_foldx']
  final_all_ddgs          : 61 features; stability=['ddg_esm2', 'ddg_struct', 'ddg_rasp', 'ddg_foldx']


## 1. Load and align all tables

In [2]:
model_df = pd.read_csv(FEATURES_CSV)

required_base = ["KEY", "Gene", "Mutation_used", "source", "Mislocalized"]
missing_base = [column for column in required_base + ESM_COLS + STRUCT_COLS
                if column not in model_df.columns]
assert not missing_base, f"Missing columns in features_v3.csv: {missing_base[:10]}"
assert len(model_df) == 2179, f"Expected 2179 rows, found {len(model_df)}"
assert model_df["KEY"].notna().all()
assert model_df["KEY"].is_unique
assert model_df["Mislocalized"].isin([0, 1]).all()
assert int(model_df["Mislocalized"].sum()) == 236

ddg_files = {
    "ddg_esm2": BASE / "ddg_esm2.csv",
    "ddg_struct": BASE / "ddg_struct.csv",
    "ddg_rasp": BASE / "ddg_rasp.csv",
    "ddg_foldx": BASE / "ddg_foldx.csv",
}

for feature_name, path in ddg_files.items():
    ddg_df = pd.read_csv(path)
    assert "KEY" in ddg_df.columns and feature_name in ddg_df.columns
    assert ddg_df["KEY"].notna().all()
    assert ddg_df["KEY"].is_unique, f"Duplicate KEY values in {path.name}"
    unknown_keys = set(ddg_df["KEY"]) - set(model_df["KEY"])
    assert not unknown_keys, f"Unknown KEY values in {path.name}: {list(unknown_keys)[:5]}"
    model_df = model_df.merge(
        ddg_df[["KEY", feature_name]],
        on="KEY", how="left", validate="one_to_one",
    )
    print(f"{feature_name:12s}: {model_df[feature_name].notna().sum()}/{len(model_df)} available")

am_df = pd.read_csv(ALPHAMISSENSE_CSV)
assert am_df["KEY"].notna().all()
assert am_df["KEY"].is_unique
model_df = model_df.merge(
    am_df[["KEY", "final_alphamissense_score", "alphamissense_status"]],
    on="KEY", how="left", validate="one_to_one",
)

assert len(model_df) == 2179
assert model_df["KEY"].is_unique
assert model_df["final_alphamissense_score"].dropna().between(0, 1).all()
print(f"AlphaMissense available: {model_df['final_alphamissense_score'].notna().sum()}/{len(model_df)}")
print(f"Binary target: n={len(model_df)}, positives={int(model_df['Mislocalized'].sum())}, "
      f"prevalence={model_df['Mislocalized'].mean():.4f}")

ddg_esm2    : 2179/2179 available
ddg_struct  : 2168/2179 available
ddg_rasp    : 2168/2179 available
ddg_foldx   : 2166/2179 available
AlphaMissense available: 2140/2179
Binary target: n=2179, positives=236, prevalence=0.1083


## 2. Prepare arrays and grouped folds

In [3]:
X_esm = model_df[ESM_COLS].to_numpy(dtype=np.float32)
X_struct = model_df[STRUCT_COLS].to_numpy(dtype=np.float32)
X_ddg = {name: model_df[[name]].to_numpy(dtype=np.float32) for name in DDG_COLS}
y = model_df["Mislocalized"].astype(int).to_numpy()
groups = model_df["Gene"].astype(str).to_numpy()

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
splits = list(cv.split(X_esm, y, groups))

fold_id = np.full(len(model_df), -1, dtype=int)
for fold, (train_idx, test_idx) in enumerate(splits):
    fold_id[test_idx] = fold
    train_genes = set(groups[train_idx])
    test_genes = set(groups[test_idx])
    assert train_genes.isdisjoint(test_genes)
    print(f"Fold {fold}: train={len(train_idx)}, test={len(test_idx)}, "
          f"test_pos={int(y[test_idx].sum())}, test_genes={len(test_genes)}")

assert (fold_id >= 0).all()
assert np.bincount(fold_id, minlength=N_SPLITS).sum() == len(model_df)

Fold 0: train=1773, test=406, test_pos=47, test_genes=168
Fold 1: train=1749, test=430, test_pos=41, test_genes=177
Fold 2: train=1748, test=431, test_pos=51, test_genes=169
Fold 3: train=1725, test=454, test_pos=63, test_genes=181
Fold 4: train=1721, test=458, test_pos=34, test_genes=176


## 3. Generate out-of-fold predictions

Balanced sample weights are the only class-imbalance treatment used during OOF evaluation. `scale_pos_weight` is not combined with these weights.

In [4]:
def make_xgb():
    return XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.5,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    )

oof = {name: np.full(len(y), np.nan, dtype=np.float32) for name in CONFIG_FEATURES}
fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(splits):
    y_train = y[train_idx]
    y_test = y[test_idx]

    esm_imputer = SimpleImputer(strategy="median")
    esm_scaler = StandardScaler()
    X_esm_train = esm_scaler.fit_transform(
        esm_imputer.fit_transform(X_esm[train_idx])
    ).astype(np.float32)
    X_esm_test = esm_scaler.transform(
        esm_imputer.transform(X_esm[test_idx])
    ).astype(np.float32)

    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    X_pca_train = pca.fit_transform(X_esm_train).astype(np.float32)
    X_pca_test = pca.transform(X_esm_test).astype(np.float32)

    struct_imputer = SimpleImputer(strategy="median")
    struct_scaler = StandardScaler()
    X_struct_train = struct_scaler.fit_transform(
        struct_imputer.fit_transform(X_struct[train_idx])
    ).astype(np.float32)
    X_struct_test = struct_scaler.transform(
        struct_imputer.transform(X_struct[test_idx])
    ).astype(np.float32)

    X_base_train = np.hstack([X_pca_train, X_struct_train]).astype(np.float32)
    X_base_test = np.hstack([X_pca_test, X_struct_test]).astype(np.float32)

    ddg_train = {}
    ddg_test = {}
    for feature_name in DDG_COLS:
        ddg_imputer = SimpleImputer(strategy="median")
        ddg_train[feature_name] = ddg_imputer.fit_transform(
            X_ddg[feature_name][train_idx]
        ).astype(np.float32)
        ddg_test[feature_name] = ddg_imputer.transform(
            X_ddg[feature_name][test_idx]
        ).astype(np.float32)

    sample_weight = compute_sample_weight("balanced", y_train)

    for config_name, stability_features in CONFIG_FEATURES.items():
        train_parts = [X_base_train] + [ddg_train[name] for name in stability_features]
        test_parts = [X_base_test] + [ddg_test[name] for name in stability_features]
        X_train = np.hstack(train_parts).astype(np.float32)
        X_test = np.hstack(test_parts).astype(np.float32)
        assert X_train.shape[1] == CONFIG_DIMS[config_name]

        model = make_xgb()
        model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
        prediction = model.predict_proba(X_test)[:, 1]
        oof[config_name][test_idx] = prediction

        fold_metrics.append({
            "scope": "fold",
            "config": config_name,
            "fold": fold,
            "n": len(test_idx),
            "n_positive": int(y_test.sum()),
            "prevalence": float(y_test.mean()),
            "auroc": roc_auc_score(y_test, prediction),
            "auprc": average_precision_score(y_test, prediction),
        })

    fold_line = [f"{name}={roc_auc_score(y_test, oof[name][test_idx]):.4f}"
                 for name in CONFIG_FEATURES]
    print(f"Fold {fold}: " + "  ".join(fold_line))

for name, predictions in oof.items():
    assert np.isfinite(predictions).all(), f"Incomplete OOF predictions for {name}"
    assert np.logical_and(predictions >= 0, predictions <= 1).all()

Fold 0: baseline_pca_struct=0.5992  plus_ddg_esm2=0.6219  plus_structure_ddgs=0.6318  final_all_ddgs=0.6459
Fold 1: baseline_pca_struct=0.6103  plus_ddg_esm2=0.5981  plus_structure_ddgs=0.5985  final_all_ddgs=0.5959
Fold 2: baseline_pca_struct=0.5674  plus_ddg_esm2=0.5947  plus_structure_ddgs=0.5699  final_all_ddgs=0.5924
Fold 3: baseline_pca_struct=0.5575  plus_ddg_esm2=0.5925  plus_structure_ddgs=0.5904  final_all_ddgs=0.6416
Fold 4: baseline_pca_struct=0.6613  plus_ddg_esm2=0.6792  plus_structure_ddgs=0.6615  final_all_ddgs=0.6976


## 4. Full-cohort and paired AlphaMissense evaluation

In [5]:
metric_rows = list(fold_metrics)
full_results = {}
for config_name, predictions in oof.items():
    result = {
        "scope": "full_oof",
        "config": config_name,
        "fold": np.nan,
        "n": len(y),
        "n_positive": int(y.sum()),
        "prevalence": float(y.mean()),
        "auroc": roc_auc_score(y, predictions),
        "auprc": average_precision_score(y, predictions),
    }
    full_results[config_name] = result
    metric_rows.append(result)

print("\nFull-cohort OOF results")
print(f"n={len(y)}, positives={int(y.sum())}, prevalence={y.mean():.4f}")
baseline_auroc = full_results["baseline_pca_struct"]["auroc"]
for config_name in CONFIG_FEATURES:
    result = full_results[config_name]
    print(f"  {config_name:24s} dim={CONFIG_DIMS[config_name]:2d}  "
          f"AUROC={result['auroc']:.4f}  AUPRC={result['auprc']:.4f}  "
          f"delta_AUROC={result['auroc'] - baseline_auroc:+.4f}")

paired_mask = model_df["final_alphamissense_score"].notna().to_numpy()
y_paired = y[paired_mask]
am_paired = model_df.loc[paired_mask, "final_alphamissense_score"].to_numpy(dtype=float)
misfit_paired = oof[FINAL_CONFIG][paired_mask]

am_result = {
    "scope": "paired_alphamissense",
    "config": "alphamissense",
    "fold": np.nan,
    "n": int(paired_mask.sum()),
    "n_positive": int(y_paired.sum()),
    "prevalence": float(y_paired.mean()),
    "auroc": roc_auc_score(y_paired, am_paired),
    "auprc": average_precision_score(y_paired, am_paired),
}
misfit_paired_result = {
    "scope": "paired_alphamissense",
    "config": FINAL_CONFIG,
    "fold": np.nan,
    "n": int(paired_mask.sum()),
    "n_positive": int(y_paired.sum()),
    "prevalence": float(y_paired.mean()),
    "auroc": roc_auc_score(y_paired, misfit_paired),
    "auprc": average_precision_score(y_paired, misfit_paired),
}
metric_rows.extend([am_result, misfit_paired_result])

print("\nPaired AlphaMissense comparison")
print(f"n={len(y_paired)}, positives={int(y_paired.sum())}, prevalence={y_paired.mean():.4f}")
print(f"  AlphaMissense: AUROC={am_result['auroc']:.4f}, AUPRC={am_result['auprc']:.4f}")
print(f"  MISFIT final:  AUROC={misfit_paired_result['auroc']:.4f}, "
      f"AUPRC={misfit_paired_result['auprc']:.4f}")
print(f"  Paired delta:  AUROC={misfit_paired_result['auroc'] - am_result['auroc']:+.4f}, "
      f"AUPRC={misfit_paired_result['auprc'] - am_result['auprc']:+.4f}")


Full-cohort OOF results
n=2179, positives=236, prevalence=0.1083
  baseline_pca_struct      dim=57  AUROC=0.5898  AUPRC=0.1567  delta_AUROC=+0.0000
  plus_ddg_esm2            dim=58  AUROC=0.6112  AUPRC=0.1688  delta_AUROC=+0.0214
  plus_structure_ddgs      dim=60  AUROC=0.6038  AUPRC=0.1697  delta_AUROC=+0.0140
  final_all_ddgs           dim=61  AUROC=0.6286  AUPRC=0.1758  delta_AUROC=+0.0388

Paired AlphaMissense comparison
n=2140, positives=235, prevalence=0.1098
  AlphaMissense: AUROC=0.6491, AUPRC=0.1619
  MISFIT final:  AUROC=0.6311, AUPRC=0.1780
  Paired delta:  AUROC=-0.0179, AUPRC=+0.0160


## 5. Save OOF predictions and metrics

In [6]:
oof_table = model_df[[
    "KEY", "Gene", "Mutation_used", "source", "Mislocalized",
    "final_alphamissense_score", "alphamissense_status",
]].copy()
oof_table["fold"] = fold_id
for config_name, predictions in oof.items():
    oof_table[f"oof_{config_name}"] = predictions
oof_table["paired_alphamissense"] = paired_mask

assert oof_table["KEY"].is_unique
assert len(oof_table) == 2179
oof_table.to_csv(OOF_CSV, index=False)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_CSV, index=False)

print(f"Saved OOF predictions: {OOF_CSV}")
print(f"Saved metrics:         {METRICS_CSV}")

Saved OOF predictions: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task12_ddg_ablation_oof.csv
Saved metrics:         /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task12_ddg_ablation_metrics.csv


## 6. Descriptive final-model feature importance

This full-data fit is used only to rank features. It is not a performance estimate.

In [7]:
esm_imputer = SimpleImputer(strategy="median")
esm_scaler = StandardScaler()
X_esm_all = esm_scaler.fit_transform(esm_imputer.fit_transform(X_esm)).astype(np.float32)
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca_all = pca.fit_transform(X_esm_all).astype(np.float32)

struct_imputer = SimpleImputer(strategy="median")
struct_scaler = StandardScaler()
X_struct_all = struct_scaler.fit_transform(
    struct_imputer.fit_transform(X_struct)
).astype(np.float32)

parts = [X_pca_all, X_struct_all]
for feature_name in DDG_COLS:
    imputer = SimpleImputer(strategy="median")
    parts.append(imputer.fit_transform(X_ddg[feature_name]).astype(np.float32))
X_final_all = np.hstack(parts).astype(np.float32)
assert X_final_all.shape == (2179, 61)

final_model = make_xgb()
final_weights = compute_sample_weight("balanced", y)
final_model.fit(X_final_all, y, sample_weight=final_weights, verbose=False)

feature_names = ([f"PC{i + 1}" for i in range(N_COMPONENTS)]
                 + STRUCT_COLS + DDG_COLS)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
importance_df["rank"] = np.arange(1, len(importance_df) + 1)
importance_df.to_csv(IMPORTANCE_CSV, index=False)

print(importance_df.head(20).to_string(index=False))
print("\nStability-related feature ranks:")
print(importance_df[importance_df["feature"].isin(DDG_COLS)].to_string(index=False))
print(f"Saved descriptive feature importance: {IMPORTANCE_CSV}")

 feature  importance  rank
ddg_esm2    0.036686     1
ddg_rasp    0.022989     2
    PC36    0.022246     3
    PC17    0.020538     4
    PC11    0.020229     5
   plddt    0.019842     6
    PC30    0.019444     7
    PC50    0.019444     8
    PC33    0.019047     9
     PC1    0.018988    10
    PC20    0.018654    11
    PC37    0.018505    12
    PC38    0.018335    13
     PC4    0.018235    14
    PC42    0.018071    15
    PC28    0.017982    16
     PC5    0.017796    17
    PC35    0.017696    18
     PC6    0.017694    19
    PC25    0.017494    20

Stability-related feature ranks:
   feature  importance  rank
  ddg_esm2    0.036686     1
  ddg_rasp    0.022989     2
 ddg_foldx    0.016574    26
ddg_struct    0.016096    31
Saved descriptive feature importance: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task12_final_feature_importance.csv
